# Llama 3.2 3B — Mental Health Counselor Fine-Tuning (QLoRA)

> **Run this notebook on Google Colab.**  
> Before running: go to **Runtime → Change runtime type → GPU** (T4 is free and sufficient).

This notebook fine-tunes `meta-llama/Llama-3.2-3B-Instruct` on the [Counsel Chat](https://huggingface.co/datasets/nbertagnolli/counsel-chat) dataset — real therapist responses to mental health questions — using 4-bit QLoRA so it fits on a single free Colab GPU.

**End-to-end workflow:**
1. Setup & GPU check
2. HuggingFace authentication
3. Validate dataset
4. Evaluate base model (before)
5. Train
6. Evaluate fine-tuned model (after)
7. Compare results
8. Push adapter to HuggingFace Hub
9. Run inference

## 1. Setup

In [ ]:
# Verify GPU is available — do not skip this
!nvidia-smi

In [ ]:
# Clone the repo and move into it
!git clone https://github.com/glenlouis8/llama-3.2-3b-alpaca-qlora.git
%cd llama-3.2-3b-alpaca-qlora

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

## 2. HuggingFace Authentication

You need two things before proceeding:
- A HuggingFace account with access to `meta-llama/Llama-3.2-3B-Instruct` — accept the license at [huggingface.co/meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
- A HuggingFace token — get one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

Add it as a Colab Secret named `HF_TOKEN`: click the **key icon** in the left sidebar → **Add new secret**.

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Write to .env so scripts can pick it up via load_dotenv()
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={os.environ['HF_TOKEN']}\n")

print("HF_TOKEN set.")

## 3. Validate Dataset

Dry-run: downloads the dataset, previews formatted samples, and checks token length distribution. No GPU needed.

In [ ]:
!python scripts/prepare_data.py

## 4. Evaluate Base Model (Before Training)

Measures perplexity and ROUGE-L on the held-out eval split **before** any fine-tuning. This is the baseline to beat.

> Takes ~5 minutes on T4.

In [ ]:
!python scripts/evaluate.py --stage before

## 5. Train

Fine-tunes the model using QLoRA (4-bit NF4 quantization + LoRA adapters on all 7 projection layers).

**Config:** 3 epochs over ~2,400 training examples, effective batch size 16, learning rate 2e-4.

> Takes ~20 minutes on T4.

In [ ]:
!python scripts/train.py

## 6. Evaluate Fine-Tuned Model (After Training)

Same metrics on the same held-out split as Step 4.

> Takes ~5 minutes on T4.

In [ ]:
!python scripts/evaluate.py --stage after

## 7. Compare Results

In [ ]:
!python scripts/evaluate.py --compare

## 8. Push Adapter to HuggingFace Hub

Uploads the LoRA adapter weights (~80MB) and tokenizer to your HuggingFace repo.

Make sure `hub.repo_id` in `configs/train_config.yaml` is set to your HuggingFace username before running.

In [ ]:
# Optional: update the hub repo_id if you haven't already
import yaml

with open("configs/train_config.yaml") as f:
    cfg = yaml.safe_load(f)

print("Current hub.repo_id:", cfg["hub"]["repo_id"])
print("If this is wrong, edit configs/train_config.yaml before running the next cell.")

In [ ]:
!python scripts/push_to_hub.py

## 9. Inference

Try the fine-tuned model on a few mental health questions.

In [ ]:
schema = "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT, hire_date DATE)"
prompt = f"List all employees in the Engineering department ordered by salary descending.\n\nSchema:\n{schema}"

!python scripts/infer.py --prompt "{prompt}"

In [ ]:
schema = "CREATE TABLE orders (id INT, customer_id INT, product TEXT, amount FLOAT, order_date DATE)"
prompt = f"What were the top 5 products by total revenue last month?\n\nSchema:\n{schema}"

print("=" * 60)
print("BASE MODEL")
print("=" * 60)
!python scripts/infer.py --base --prompt "{prompt}"

print("\n" + "=" * 60)
print("FINE-TUNED MODEL")
print("=" * 60)
!python scripts/infer.py --prompt "{prompt}"

In [ ]:
your_question = "How many users signed up each month this year?"  # @param {type:"string"}
your_schema = "CREATE TABLE users (id INT, name TEXT, email TEXT, signup_date DATE)"  # @param {type:"string"}

prompt = f"{your_question}\n\nSchema:\n{your_schema}"
!python scripts/infer.py --prompt "{prompt}"

In [ ]:
from google.colab import userdata

github_token = userdata.get("GITHUB_TOKEN")

# Inject token into remote URL so git can push without interactive login
!git remote set-url origin https://{github_token}@github.com/glenlouis8/llama-3.2-3b-alpaca-qlora.git

# Configure git identity (required for commits)
!git config user.email "glenlouis@users.noreply.github.com"
!git config user.name "glenlouis8"

# Commit and push the evaluation results
!git add results/before_finetune.json results/after_finetune.json
!git commit -m "Add evaluation results"
!git push

## 10. Push Results to GitHub

Add your GitHub token as a Colab Secret named `GITHUB_TOKEN`: click the **key icon** in the left sidebar → **Add new secret**.

To generate a token: go to [github.com/settings/tokens](https://github.com/settings/tokens) → **Generate new token (classic)** → check the `repo` scope.